# Pandas Data Analysis: US Domestic Air Traffic

 
**Author:** Husnain Maroof  
**Institution:** University of Engineering & Technology, Lahore  
**Department:** Mechatronics and Control Engineering  
**GitHub:** https://github.com/husnainalix77  
**Date:** May 05-12, 2026




## Project Overview

This project analyses a real-world dataset of **US domestic air traffic (January–May 2013)**
containing over 136,000 flight segments across the United States.

The analysis covers:
- DataFrame merging across multiple relational tables
- GroupBy aggregation and passenger trend analysis
- Set operations for multi-airport route analysis
- String filtering for manufacturer comparisons
- Hub-and-spoke network connectivity analysis

**Tools Used:** Python, Pandas

## Dataset Description

| File | Rows | Key Columns |
|------|------|-------------|
| `flights.json` | 136,012 | airline_id, from_id, to_id, aircraft_id, distance, airtime, passengers, month |
| `airlines.json` | 1,559 | id, code, name |
| `airports.json` | 6,267 | id, code, name |
| `aircrafts.json` | 384 | id, name |

### Relational Structure
- `flights.airline_id` → `airlines.id`
- `flights.from_id` / `flights.to_id` → `airports.id`
- `flights.aircraft_id` → `aircrafts.id`

## Data Loading — Starter Code

In [1]:
import pandas as pd

# Load all JSON Files
flights_df = pd. read_json ("flights.json")
airlines_df = pd.read_json("airlines.json")
aircrafts_df = pd.read_json("aircrafts.json")
airports_df = pd.read_json("airports.json")

# Quick sanity check
print("flights_df   :", flights_df.shape)
print("airlines_df  :", airlines_df.shape)
print("aircrafts_df :", aircrafts_df.shape)
print("airports_df  :", airports_df.shape)
print (flights_df.head(3))

flights_df   : (136012, 8)
airlines_df  : (1559, 3)
aircrafts_df : (384, 2)
airports_df  : (6267, 3)
  airline_id from_id to_id aircraft_id distance airtime passengers month
0         59     966  1482          17       37      16          1     1
1         59     966  2241          17       20      67          2     1
2         59     966  2788          17       28      18          0     1


## Task 1. Departures from Nashville International

In [19]:
# Merge from_id in flights_df with id in airports_df
departures_merged = pd.merge(flights_df, airports_df, left_on="from_id", right_on="id")

# Filter for Nashville International (IATA code: BNA)
bna_departures = departures_merged[departures_merged["code"] == "BNA"]

# Display the result
print(f"Total flight segments departing from Nashville International (BNA): {len(bna_departures)}")

Total flight segments departing from Nashville International (BNA): 925


## Task 2. Flights Operated by Boeing 737-800

In [ ]:
flights_with_aircraft = pd.merge(flights_df, aircrafts_df, left_on = "aircraft_id", right_on = "id")
boeing_737_800 = flights_with_aircraft[flights_with_aircraft["name"] == "Boeing 737-800"]

# Display the result
print(f"Total flight segments operated by Boeing 737-800: {len(boeing_737_800)}")

Total flight segments operated by Boeing 737-800: 8996


## Task 3. Maximum Passengers in a Single Segment

In [5]:
# Locate row index with maximum number of passengers
max_idx = pd.to_numeric(flights_df["passengers"], errors = "coerce").idxmax()
max_flight = flights_df.loc[[max_idx]] # keep it as dataframe

# Merge airline name
max_flight = pd.merge(max_flight, airlines_df[["id", "name"]], left_on= "airline_id", right_on="id")
max_flight = max_flight.rename(columns={"name": "airline_name"}).drop(columns=["id"])

# Merge departure airport
max_flight = pd.merge(max_flight, airports_df[["id", "name"]], left_on= "from_id", right_on= "id")
max_flight = max_flight.rename(columns={"name": "departure_airport"}).drop(columns=["id"])

# Merge arrival airport
max_flight = pd.merge(max_flight, airports_df[["id", "name"]], left_on= "to_id", right_on= "id")
max_flight = max_flight.rename(columns={"name": "arrival_airport"}).drop(columns=["id"])

# Merge aircraft model
max_flight = pd.merge(max_flight, aircrafts_df, left_on= "aircraft_id", right_on= "id")
max_flight = max_flight.rename(columns={"name": "aircraft_model"}).drop(columns=["id"])

# Display the result
print(f"Maximum Passengers in a Single Segment: {int(max_flight['passengers'].values[0])}")
print(f"Departure Airport : {max_flight['departure_airport'].values[0]}")
print(f"Arrival Airport   : {max_flight['arrival_airport'].values[0]}")
print(f"Airline           : {max_flight['airline_name'].values[0]}")
print(f"Aircraft Model    : {max_flight['aircraft_model'].values[0]}")
print(f"Month             : {max_flight['month'].values[0]}")

Maximum Passengers in a Single Segment: 82812
Departure Airport : Kahului, HI: Kahului Airport
Arrival Airport   : Honolulu, HI: Honolulu International
Airline           : Hawaiian Airlines Inc.
Aircraft Model    : Boeing 717-200
Month             : 5


## Task 4. Longest Flight Segment by Distance

In [25]:
# Locate row index with largest distance
max_dist = pd.to_numeric(flights_df["distance"], errors="coerce").idxmax()
max_dist_flight = flights_df.loc[[max_dist]]

# Merge departure airport
dept_airport = pd.merge(max_dist_flight, airports_df[["id", "name"]], left_on= "from_id", right_on= "id")

# Merge arrival airport
arrival_airport = pd.merge(max_dist_flight, airports_df[["id", "name"]], left_on= "to_id", right_on= "id") 

# Display the result
print(f"Distance:          {max_dist_flight['distance'].values[0]} miles")
print(f"Departure Airport: {dept_airport['name'].values[0]}")
print(f"Arrival Airport:   {arrival_airport['name'].values[0]}")

Distance:          4983 miles
Departure Airport: Honolulu, HI: Honolulu International
Arrival Airport:   New York, NY: John F. Kennedy International


## Task 5. Airline with the Most Flight Segments

In [ ]:
# Merge flights_df with airlines_df
merged = pd.merge(flights_df, airlines_df, left_on= "airline_id", right_on= "id")

# Group by airline name and count size 
segments_per_airline = merged.groupby("name").size()

# Find maximum flight segments
max_airline = segments_per_airline.idxmax()
max_count = segments_per_airline.max()

# Display the result
print(f"Airline with most flight segments: {max_airline}")
print(f"Total flight segments: {max_count}")

Airline with most flight segments: Southwest Airlines Co.
Total flight segments: 11959


## Task 6. Total Passengers per Month

In [24]:
monthly_passengers = flights_df.groupby("month")["passengers"].sum()

# Display the result
print("Total Passengers per Month:")
print(monthly_passengers)

Total Passengers per Month:
month
1    48833734
2    46621722
3    57682473
4    54299772
5    57729792
             
Name: passengers, dtype: object


### Interpretation — Monthly Passenger Trend

March and May have the most passengers carried (approximately 57.7 million each), while February records the lowest total because of being shortest month of year resulting in less availability of flights. The surge in March is likely driven by spring break travel, while surge in May is just due to start of summer holidays and end-of-semester travel. 

## Task 7. Airlines Serving Orlando (MCO) as Destination

In [ ]:
# Merge flights_df with airports_df
flights_with_airports = pd.merge(flights_df[["to_id", "airline_id"]], airports_df, left_on= "to_id", right_on= "id")
flights_with_airports = flights_with_airports.rename(columns={"name": "airport_name", "id": "airport_id"})

# Filter for code = "MCO"
mco_flights = flights_with_airports.query("code == 'MCO'")

# Merge with airlines_df to get names
mco_airlines = pd.merge(mco_flights, airlines_df[["id", "name"]], left_on= "airline_id", right_on= "id")
unique_airlines = mco_airlines["name"].unique()

# Display the result
print(f"Total airlines arriving at Orlando (MCO): {len(unique_airlines)}")
print(pd.Series(unique_airlines).to_string(index=False)) # this removes the brackets and formats it nicely 

Total airlines arriving at Orlando (MCO): 29
                                 Endeavor Air Inc.
                            American Airlines Inc.
                              Alaska Airlines Inc.
                                   JetBlue Airways
US Airways Inc. (Merged with America West 9/05....
                                    Virgin America
                             United Parcel Service
                                     ABX Air, Inc.
                               Ameristar Air Cargo
                              Delta Air Lines Inc.
                          ExpressJet Airlines Inc.
                       AirTran Airways Corporation
                                     Allegiant Air
                           Miami Air International
                                  Spirit Air Lines
                             Shuttle America Corp.
            Sun Country Airlines d/b/a MN Airlines
                             United Air Lines Inc.
                            Southwest

## Task 8. Aircraft Types Departing from Seattle (SEA)

In [6]:
# Merge flights_df with airports_df
flights_with_airports = pd.merge(flights_df[["from_id", "aircraft_id"]], airports_df, left_on= "from_id", right_on= "id")
flights_with_airports = flights_with_airports.rename(columns={"name": "airport_name", "id": "airport_id"})

# Filter for code = "SEA"
sea_flights = flights_with_airports.query("code == 'SEA'")

# Merge with aircrafts_df to get unique names
sea_aircrafts = pd.merge(sea_flights, aircrafts_df[["id", "name"]], left_on= "aircraft_id", right_on= "id")
unique_aircrafts = sea_aircrafts["name"].unique()

# Display the result
print(f"Aircraft Models operating departure from Seattle/Tacoma International :")
print(pd.Series(unique_aircrafts).to_string(index=False)) 

Aircraft Models operating departure from Seattle/Tacoma International :
                              Boeing 737-800
McDonnell Douglas DC9 Super 80/MD81/82/83/88
                              Boeing 757-200
                        Boeing 737-700/700LR
                              Boeing 737-400
                              Boeing 737-900
               Airbus Industrie A320-100/200
                        Boeing 767-300/300ER
                       Airbus Industrie A321
                       Airbus Industrie A319
                        Boeing 767-200/ER/EM
                              Boeing 757-300
                           Boeing 767-400/ER
                             Airbus A330-300
                              Boeing 747-400
                     McDonnell Douglas MD-90
                          Cessna 208 Caravan
                         Boeing 727-200/231A
                   Canadair RJ-200ER /RJ-440
                             Canadair RJ-700
                    Embraer 

## Task 9. Airlines Serving Both JFK and LAX

In [7]:
jfk_id = airports_df[airports_df["code"] == "JFK"]["id"].values[0]
lax_id = airports_df[airports_df["code"] == "LAX"]["id"].values[0]

airlines_to_jfk = set(flights_df[flights_df["to_id"] == jfk_id]["airline_id"])
airlines_to_lax = set(flights_df[flights_df["to_id"] == lax_id]["airline_id"])

airlines_both = airlines_to_jfk & airlines_to_lax

names = pd.merge(pd.DataFrame(airlines_both, columns=["airline_id"]), 
                 airlines_df, 
                 left_on="airline_id", 
                 right_on="id")

print(f"Total airlines serving both JFK and LAX: {len(names)}")
print(names["name"].to_string(index=False))

Total airlines serving both JFK and LAX: 19
                      American Eagle Airlines Inc.
                                 Southern Air Inc.
                                   Kalitta Air LLC
                                    Virgin America
                            American Airlines Inc.
                            Hawaiian Airlines Inc.
                                     ABX Air, Inc.
                           North American Airlines
                                   JetBlue Airways
                             USA Jet Airlines Inc.
                             United Air Lines Inc.
                             United Parcel Service
US Airways Inc. (Merged with America West 9/05....
                       Federal Express Corporation
                                    Atlas Air Inc.
                              Delta Air Lines Inc.
                           Miami Air International
            Sun Country Airlines d/b/a MN Airlines
                                 Repub

## Task 10. High-Traffic Airports (>1,000 Segments)

In [15]:
# Stack from_id and to_id from airports_df into a single Series
all_connections = pd.concat([flights_df["from_id"], flights_df["to_id"]])

# Total flight counts for each airportID
airports_counts = all_connections.value_counts()

# Filter for counts > 1000
high_traffic = airports_counts[airports_counts > 1000]

# Convert Series to DataFrame and reset index
high_traffic_df = high_traffic.reset_index()
high_traffic_df.columns = ["airport_id", "segment_count"]

# Merge with airports_df to get airport names
result = pd.merge(high_traffic_df, airports_df, left_on= "airport_id", right_on= "id").drop(columns=["id"])

# Display the result
print(f"Total airports with more than 1000 flight segments: {len(result)}")
print(result[["name", "segment_count"]].to_string(index=False))


Total airports with more than 1000 flight segments: 58
                                                               name  segment_count
              Atlanta, GA: Hartsfield-Jackson Atlanta International           8489
                         Chicago, IL: Chicago O''Hare International           8243
                                   Denver, CO: Denver International           7643
                 Minneapolis, MN: Minneapolis-St Paul International           6670
                            Detroit, MI: Detroit Metro Wayne County           5774
                     Charlotte, NC: Charlotte Douglas International           5506
                  Houston, TX: George Bush Intercontinental/Houston           5228
             Dallas/Fort Worth, TX: Dallas/Fort Worth International           5130
                       Philadelphia, PA: Philadelphia International           4516
                                 Memphis, TN: Memphis International           4508
                         Los Ang

## Task 11. Airbus vs. Boeing: Segment Counts by Manufacturer

In [12]:
# Merge flights_df with aircrafts_df
flights_with_aircraft = pd.merge(flights_df, aircrafts_df, left_on= "aircraft_id", right_on = "id")

# Filter out flights that operate with Airbus and Boeing
airbus_mask = flights_with_aircraft["name"].str.contains("Airbus")
boeing_mask = flights_with_aircraft["name"].str.contains("Boeing")

# Display the result
print(f"Flight segments operated by Airbus aircraft: {sum(airbus_mask)}")
print(f"Flight segments operated by Boeing aircraft: {sum(boeing_mask)}")


Flight segments operated by Airbus aircraft: 18053
Flight segments operated by Boeing aircraft: 38588


## Task 20. Airport Hub Connectivity Analysis

### (a) Airport with the Most Direct Connections

In [ ]:
departure = flights_df[["from_id","to_id"]].rename(
    columns= {"from_id": "airport", "to_id": "partner"}
)

arrival = flights_df[["to_id","from_id"]].rename(
    columns= {"to_id": "airport", "from_id": "partner"}
)

all_connections = pd.concat([departure, arrival]).drop_duplicates()

connectivity = all_connections.groupby("airport")["partner"].nunique() # Series (counts unique values)
hub_id = connectivity.idxmax() # hub id
connectivity_df = connectivity.reset_index() # Convert back to data frame
connectivity_df.columns = ["airport_id", "connections"]
result = pd.merge(connectivity_df, airports_df, left_on="airport_id", right_on="id")
hub_row = result[result["connections"] == result["connections"].max()]
print(hub_row[["name", "connections"]].to_string(index=False))

airport
3       6
6       2
9       8
11      1
13      3
       ..
6242    1
6251    7
6256    1
6257    3
        1
Name: partner, Length: 1000, dtype: int64


### (b) Top Partner of the Hub

In [ ]:
# Those rows only where flight goes to hub or flight comes from hub
hub_flights = flights_df[(flights_df["to_id"] == hub_id) | (flights_df["from_id"] == hub_id)]
partners = pd.concat([
    hub_flights[hub_flights["from_id"] == hub_id]["to_id"], # All destinations where flight starts from hub
    hub_flights[hub_flights["to_id"] == hub_id]["from_id"]  # All sources where flight comes to hub
])

partner_counts = partners.value_counts()

top_partner_id = partner_counts.idxmax()
top_partner_name = airports_df[airports_df["id"] == top_partner_id]["name"].values[0]
top_partner_count = partner_counts.max()

print(f"Top Partner Airport: {top_partner_name}")
print(f"Total Segments with Hub: {top_partner_count}")


Top Partner Airport: Chicago, IL: Chicago O''Hare International
Total Segments with Hub: 249


### Interpretation — Airport Hub Connectivity Analysis

Minneapolis–St Paul International (MSP) is the most connected airport in the
US domestic network, linked to 202 unique airports. This shows that the US air
network works like a hub-and-spoke system — a few major airports handle most
of the connections while smaller airports depend on them for routes. MSP's
busiest partner is Chicago O'Hare (ORD) with 249 combined flight segments,
which makes sense as both are major Midwest cities with heavy business travel.
In general, hub airports tend to be in large cities that are geographically
central, making them convenient transfer points for passengers travelling
across the country.